# 任务 4：用户行为转化漏斗分析

## 任务描述
统计电商用户行为转化路径的各阶段用户数及转化率

## 转化路径定义
**PV（浏览）** -> **Fav（收藏）/Cart（加购）** -> **Buy（购买）**

## 指标计算
1. 有多少独立用户产生了 PV？
2. 这些 PV 用户中，有多少进一步产生了 Fav 或 Cart 行为？
3. 最终有多少完成了 Buy？

输出各阶段的绝对用户数以及阶段转化率（Conversion Rate）

In [ ]:
import polars as pl
import time

print(f"Polars 版本：{pl.__version__}")

In [ ]:
# 配置
INPUT_PATH = "clean_data_deduped.parquet"  # 使用已去重的数据

# 性能计时开始
start_time = time.time()

## 1. 加载数据（Lazy 模式）

In [ ]:
q = pl.scan_parquet(INPUT_PATH)
print("数据加载完成")

## 2. 查看行为类型

In [ ]:
behavior_types = q.select(pl.col("behavior_type").unique()).collect()
print("行为类型:", behavior_types["behavior_type"].to_list())

## 3. 阶段 1：统计产生 PV 行为的独立用户

In [ ]:
pv_users = q.filter(pl.col("behavior_type") == "pv").select(
    pl.col("user_id").unique().alias("user_id")
).collect()

pv_user_count = pv_users.height
print(f"产生 PV 行为的独立用户数：{pv_user_count:,}")

## 4. 阶段 2：统计产生 Fav 或 Cart 行为的用户

In [ ]:
fav_cart_users = q.filter(
    pl.col("behavior_type").is_in(["fav", "cart"])
).select(
    pl.col("user_id").unique().alias("user_id")
).collect()

fav_cart_user_count = fav_cart_users.height
print(f"产生 Fav 或 Cart 行为的独立用户数：{fav_cart_user_count:,}")

# 计算与 PV 用户的交集（PV -> Fav/Cart 转化）
pv_user_set = set(pv_users["user_id"].to_list())
fav_cart_user_set = set(fav_cart_users["user_id"].to_list())
pv_to_fav_cart_users = pv_user_set & fav_cart_user_set
pv_to_fav_cart_count = len(pv_to_fav_cart_users)
print(f"PV 用户中进一步产生 Fav 或 Cart 行为的用户数：{pv_to_fav_cart_count:,}")

## 5. 阶段 3：统计产生 Buy 行为的用户

In [ ]:
buy_users = q.filter(pl.col("behavior_type") == "buy").select(
    pl.col("user_id").unique().alias("user_id")
).collect()

buy_user_count = buy_users.height
print(f"产生 Buy 行为的独立用户数：{buy_user_count:,}")

# 计算完整转化路径用户（PV -> Fav/Cart -> Buy）
buy_user_set = set(buy_users["user_id"].to_list())
full_funnel_users = pv_to_fav_cart_users & buy_user_set
full_funnel_count = len(full_funnel_users)
print(f"完成完整转化路径（PV -> Fav/Cart -> Buy）的用户数：{full_funnel_count:,}")

## 6. 计算转化率

In [ ]:
print("=" * 60)
print("转化率计算")
print("=" * 60)

# PV -> Fav/Cart 转化率
pv_to_fav_cart_rate = (pv_to_fav_cart_count / pv_user_count * 100) if pv_user_count > 0 else 0
print(f"PV -> Fav/Cart 转化率：{pv_to_fav_cart_rate:.2f}%")

# Fav/Cart -> Buy 转化率
fav_cart_to_buy_rate = (full_funnel_count / pv_to_fav_cart_count * 100) if pv_to_fav_cart_count > 0 else 0
print(f"Fav/Cart -> Buy 转化率：{fav_cart_to_buy_rate:.2f}%")

# 整体转化率（PV -> Buy）
overall_rate = (full_funnel_count / pv_user_count * 100) if pv_user_count > 0 else 0
print(f"整体转化率（PV -> Buy）：{overall_rate:.2f}%")

## 7. 各行为类型的详细统计

In [ ]:
print("=" * 60)
print("各行为类型的详细统计")
print("=" * 60)

behavior_stats = q.group_by("behavior_type").agg(
    pl.col("user_id").n_unique().alias("独立用户数"),
    pl.len().alias("行为次数")
).sort("behavior_type")
print(behavior_stats)

## 8. 性能统计

In [ ]:
end_time = time.time()
total_time = end_time - start_time

print("=" * 60)
print("性能统计")
print("=" * 60)
print(f"总耗时：{total_time:.2f} 秒")

## 9. 转化漏斗总结

In [ ]:
print("=" * 60)
print("转化漏斗总结")
print("=" * 60)

print(f"""
┌─────────────────────────────────────────────────────────────┐
│                    用户行为转化漏斗                          │
├─────────────────────────────────────────────────────────────┤
│  阶段 1: PV（浏览）                                            │
│  独立用户数：{pv_user_count:>12,}                                 │
│                                                             │
│  ↓ 转化率：{pv_to_fav_cart_rate:>6.2f}%                                          │
│                                                             │
│  阶段 2: Fav（收藏）/ Cart（加购）                                │
│  独立用户数：{pv_to_fav_cart_count:>12,} (仅计算同时有 PV 的用户)                  │
│                                                             │
│  ↓ 转化率：{fav_cart_to_buy_rate:>6.2f}%                                          │
│                                                             │
│  阶段 3: Buy（购买）                                          │
│  独立用户数：{full_funnel_count:>12,} (仅计算完成完整路径的用户)                  │
│                                                             │
├─────────────────────────────────────────────────────────────┤
│  整体转化率（PV -> Buy）：{overall_rate:>6.2f}%                                  │
└─────────────────────────────────────────────────────────────┘
""")

## 10. 保存结果

In [ ]:
# 创建汇总 DataFrame
summary_data = {
    "stage": ["1_PV", "2_Fav_Cart", "3_Buy_Full_Funnel"],
    "user_count": [pv_user_count, fav_cart_user_count, full_funnel_count],
    "conversion_rate": [100.0, pv_to_fav_cart_rate, overall_rate]
}
summary_df = pl.DataFrame(summary_data)
summary_df.write_csv("funnel_summary.csv")
print("汇总结果已保存至：funnel_summary.csv")

# 保存各阶段用户 ID 列表
pv_users.write_parquet("pv_users.parquet")
fav_cart_users.write_parquet("fav_cart_users.parquet")
buy_users.write_parquet("buy_users.parquet")
print("各阶段用户 ID 已保存")